In [1]:
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# Load the training data from the JSONL file
train_data = []
with open("../scicite/train.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        example = json.loads(line)
        train_data.append({
            'text': example['string'],
            'label': example['label']
        })

print(f"Training data loaded: {len(train_data)} examples")

# Load the dev data
dev_data = []
with open("../scicite/dev.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        example = json.loads(line)
        dev_data.append({
            'text': example['string'],
            'label': example['label']
        })

print(f"Dev data loaded: {len(dev_data)} examples")

# Convert to dataframe so it's easier to work with
train_df = pd.DataFrame(train_data)
dev_df = pd.DataFrame(dev_data)

print(f"Train: {train_df.shape[0]} rows")
print(f"Dev: {dev_df.shape[0]} rows")

Training data loaded: 8243 examples
Dev data loaded: 916 examples
Train: 8243 rows
Dev: 916 rows


In [2]:
# Create TF-IDF vectorizer
# This converts text sentences into numbers that the model can understand
vectorizer = TfidfVectorizer(
    max_features=1000,  # Use top 1000 most important words
    ngram_range=(1, 2),  # Look at single words and pairs of words
    lowercase=True,
    stop_words='english'  # Remove common words like "the", "a", etc
)

# Fit on training data and transform it
print("Converting text to TF-IDF vectors...")
X_train = vectorizer.fit_transform(train_df['text'])
print(f"Training vectors shape: {X_train.shape}")

# Transform dev data using the same vectorizer
X_dev = vectorizer.transform(dev_df['text'])
print(f"Dev vectors shape: {X_dev.shape}")

# Get the labels
y_train = train_df['label'].values
y_dev = dev_df['label'].values

print(f"\nTrain labels: {np.unique(y_train)}")
print(f"Dev labels: {np.unique(y_dev)}")


Converting text to TF-IDF vectors...
Training vectors shape: (8243, 1000)
Dev vectors shape: (916, 1000)

Train labels: ['background' 'method' 'result']
Dev labels: ['background' 'method' 'result']


In [3]:
# Train Naive Bayes classifier on the training data
print("Training Naive Bayes model...")
model = MultinomialNB()
model.fit(X_train, y_train)
print("Model trained!")

# Make predictions on training data
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='macro')

print(f"\nTraining Results:")
print(f"  Accuracy: {train_accuracy:.4f}")
print(f"  Macro F1: {train_f1:.4f}")

# Make predictions on dev data
y_dev_pred = model.predict(X_dev)
dev_accuracy = accuracy_score(y_dev, y_dev_pred)
dev_f1 = f1_score(y_dev, y_dev_pred, average='macro')

print(f"\nDev Results:")
print(f"  Accuracy: {dev_accuracy:.4f}")
print(f"  Macro F1: {dev_f1:.4f}")

Training Naive Bayes model...
Model trained!

Training Results:
  Accuracy: 0.7771
  Macro F1: 0.7029

Dev Results:
  Accuracy: 0.7445
  Macro F1: 0.6641


In [4]:
# Show detailed results for each class
print("Detailed Classification Report:")
print(classification_report(y_dev, y_dev_pred))

# Show which examples were misclassified
print("\nSample Misclassifications:")
misclassified_indices = np.where(y_dev_pred != y_dev)[0]
print(f"Total misclassified: {len(misclassified_indices)} out of {len(y_dev)}")

# Show 3 examples of wrong predictions
for i, idx in enumerate(misclassified_indices[:3]):
    print(f"\nExample {i+1}:")
    print(f"  Text: {dev_df.iloc[idx]['text'][:100]}...")
    print(f"  True label: {y_dev[idx]}")
    print(f"  Predicted: {y_dev_pred[idx]}")

Detailed Classification Report:
              precision    recall  f1-score   support

  background       0.73      0.91      0.81       538
      method       0.75      0.56      0.64       255
      result       0.90      0.38      0.54       123

    accuracy                           0.74       916
   macro avg       0.79      0.62      0.66       916
weighted avg       0.76      0.74      0.73       916


Sample Misclassifications:
Total misclassified: 234 out of 916

Example 1:
  Text: Our results confirm the other studies suggesting that antioxidants may have a protective effect agai...
  True label: result
  Predicted: background

Example 2:
  Text: The regions of dhfr and dhps genes containing the mutations for antifolate resistance were amplified...
  True label: method
  Predicted: background

Example 3:
  Text: This cell binding pattern was identical to that obtained with a recombinant human sCD5 (rshCD5) mole...
  True label: result
  Predicted: background


In [5]:
# Save TF-IDF results to a file for later comparison
results_tfidf = {
    'model': 'TF-IDF + Naive Bayes',
    'dev_accuracy': dev_accuracy,
    'dev_f1': dev_f1,
    'train_accuracy': train_accuracy,
    'train_f1': train_f1,
    'classification_report': classification_report(y_dev, y_dev_pred, output_dict=True)
}

# Save to JSON
import json
with open('../results_tfidf.json', 'w') as f:
    json.dump(results_tfidf, f, indent=2)

print("✓ TF-IDF results saved to results_tfidf.json")

# Also save the model itself for later use
import pickle
with open('../tfidf_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("✓ Model and vectorizer saved")

✓ TF-IDF results saved to results_tfidf.json
✓ Model and vectorizer saved
